# 🧩 Mini-Lab: Map the Pipeline Stages

**Module: Ship a Working AI Feature Today** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Name** the six stages of an AI pipeline from input to output
2. **Identify** which stage a given failure belongs to
3. **Describe** what each stage does in plain language

## Target Concepts

| Concept | Gloss |
|---------|-------|
| AI Pipeline Stages | A practical AI feature pipeline with six boxes: ingest, prepare, call, validate, log, evaluate. The model call is just one box, not the whole system. |


## The Six-Box AI Pipeline

Every AI feature you ship — whether it's a chatbot, a classifier, or a document extractor — follows the same six stages:

```
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│ 1.INGEST │──▶│2.PREPARE │──▶│ 3.PROMPT │──▶│ 4.CALL   │──▶│ 5.PARSE  │──▶│6.DELIVER │
│          │   │          │   │          │   │  MODEL   │   │  OUTPUT  │   │          │
└──────────┘   └──────────┘   └──────────┘   └──────────┘   └──────────┘   └──────────┘
```

| # | Stage | What Happens | Example Failure |
|---|-------|-------------|------------------|
| 1 | **Ingest** | Load raw data (file, API, user input) | File not found, encoding error |
| 2 | **Prepare** | Clean, chunk, or format the data | Text too long for context window |
| 3 | **Prompt** | Build the prompt (system + user + data) | Missing template variable |
| 4 | **Call Model** | Send request to LLM API | API key invalid, rate limit hit |
| 5 | **Parse Output** | Extract structured data from response | JSON parse error, missing field |
| 6 | **Deliver** | Return result to user or downstream system | Wrong format for frontend |

In [1]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Define the Pipeline Stages

Let's encode the six stages so we can reference them programmatically.

In [2]:
PIPELINE_STAGES = {
    1: {"name": "Ingest",       "does": "Load raw data from files, APIs, or user input"},
    2: {"name": "Prepare",      "does": "Clean, chunk, or format data for the model"},
    3: {"name": "Prompt",       "does": "Assemble the system prompt, user message, and context"},
    4: {"name": "Call Model",   "does": "Send the request to the LLM API and wait for a response"},
    5: {"name": "Parse Output", "does": "Extract structured data from the model's raw response"},
    6: {"name": "Deliver",      "does": "Return the result to the user or a downstream system"},
}

# Print the pipeline
header = "| # | Stage | What It Does |\n|---|-------|-------------|\n"
rows = ""
for num, info in PIPELINE_STAGES.items():
    rows += f"| {num} | **{info['name']}** | {info['does']} |\n"

md(header + rows)

| # | Stage | What It Does |
|---|-------|-------------|
| 1 | **Ingest** | Load raw data from files, APIs, or user input |
| 2 | **Prepare** | Clean, chunk, or format data for the model |
| 3 | **Prompt** | Assemble the system prompt, user message, and context |
| 4 | **Call Model** | Send the request to the LLM API and wait for a response |
| 5 | **Parse Output** | Extract structured data from the model's raw response |
| 6 | **Deliver** | Return the result to the user or a downstream system |


## Locate Failures in the Pipeline

The real skill isn't memorizing the six stages — it's **looking at an error and immediately knowing which box it belongs to**. Let's practice with realistic failure scenarios.

In [3]:
# Realistic failure scenarios an AI engineer encounters
failures = [
    {
        "description": "FileNotFoundError: [Errno 2] No such file or directory: 'data/reviews.csv'",
        "stage": 1,
        "explanation": "The data file couldn't be loaded — this is an Ingest failure."
    },
    {
        "description": "The prompt sent 150,000 tokens but the model's context window is 128K.",
        "stage": 2,
        "explanation": "The text wasn't chunked or trimmed properly — this is a Prepare failure."
    },
    {
        "description": "KeyError: 'customer_name' when building the prompt template.",
        "stage": 3,
        "explanation": "A template variable was missing — this is a Prompt failure."
    },
    {
        "description": "openai.AuthenticationError: Invalid API key provided.",
        "stage": 4,
        "explanation": "The API call itself failed — this is a Call Model failure."
    },
    {
        "description": "json.decoder.JSONDecodeError: Expecting ',' — the model returned markdown instead of JSON.",
        "stage": 5,
        "explanation": "The model's response couldn't be parsed — this is a Parse Output failure."
    },
    {
        "description": "The API returns a dict but the frontend expects a flat list of strings.",
        "stage": 6,
        "explanation": "The result format doesn't match what the consumer needs — this is a Deliver failure."
    },
]

In [4]:
def locate_failure(failure):
    """Given a failure dict, return a formatted diagnosis."""
    stage_num = failure["stage"]
    stage_name = PIPELINE_STAGES[stage_num]["name"]
    return (
        f"**Failure:** {failure['description']}\n\n"
        f"📍 **Stage {stage_num} — {stage_name}**\n\n"
        f"*Why:* {failure['explanation']}\n\n---"
    )

In [5]:
# Walk through each failure and locate it in the pipeline
output = "## Failure → Stage Mapping\n\n"
for f in failures:
    output += locate_failure(f) + "\n"

md(output)

## Failure → Stage Mapping

**Failure:** FileNotFoundError: [Errno 2] No such file or directory: 'data/reviews.csv'

📍 **Stage 1 — Ingest**

*Why:* The data file couldn't be loaded — this is an Ingest failure.

---
**Failure:** The prompt sent 150,000 tokens but the model's context window is 128K.

📍 **Stage 2 — Prepare**

*Why:* The text wasn't chunked or trimmed properly — this is a Prepare failure.

---
**Failure:** KeyError: 'customer_name' when building the prompt template.

📍 **Stage 3 — Prompt**

*Why:* A template variable was missing — this is a Prompt failure.

---
**Failure:** openai.AuthenticationError: Invalid API key provided.

📍 **Stage 4 — Call Model**

*Why:* The API call itself failed — this is a Call Model failure.

---
**Failure:** json.decoder.JSONDecodeError: Expecting ',' — the model returned markdown instead of JSON.

📍 **Stage 5 — Parse Output**

*Why:* The model's response couldn't be parsed — this is a Parse Output failure.

---
**Failure:** The API returns a dict but the frontend expects a flat list of strings.

📍 **Stage 6 — Deliver**

*Why:* The result format doesn't match what the consumer needs — this is a Deliver failure.

---


## Quick Self-Check

Let's verify you can map stages by name. The function below takes a stage name and returns its number and description.

In [6]:
def lookup_stage(name):
    """Look up a pipeline stage by name (case-insensitive)."""
    for num, info in PIPELINE_STAGES.items():
        if info["name"].lower() == name.lower():
            return f"Stage {num}: **{info['name']}** — {info['does']}"
    return f"Unknown stage: '{name}'. Valid stages: {[s['name'] for s in PIPELINE_STAGES.values()]}"

# Test a few lookups
for stage_name in ["Ingest", "Prompt", "Parse Output"]:
    md(lookup_stage(stage_name))

Stage 1: **Ingest** — Load raw data from files, APIs, or user input

Stage 3: **Prompt** — Assemble the system prompt, user message, and context

Stage 5: **Parse Output** — Extract structured data from the model's raw response

## 🎯 Summary

### Key Takeaways

1. **Six stages** — every AI feature follows the same path: Ingest → Prepare → Prompt → Call Model → Parse Output → Deliver
2. **Failure localization** — when something breaks, the first question is *"which of the six boxes did it break in?"* — not *"what's wrong with my AI?"*
3. **Stages are independent** — an API key error (Stage 4) has nothing to do with your prompt logic (Stage 3); naming the stage focuses your debugging